<a href="https://colab.research.google.com/github/VuKhang-ict/vnli-educorpus/blob/main/notebooks/04_compare_annotators_kappa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install pandas scikit-learn

In [ ]:
import json
import pandas as pd
from sklearn.metrics import cohen_kappa_score
from google.colab import files

uploaded = files.upload()
json_files = [k for k in uploaded.keys() if k.endswith(".json")]
print(json_files)

Saving project-5-at-2026-03-21-10-28-f80f4f79.json to project-5-at-2026-03-21-10-28-f80f4f79.json
Saving project-2-at-2026-03-21-10-28-5a7e97fe.json to project-2-at-2026-03-21-10-28-5a7e97fe.json
['project-5-at-2026-03-21-10-28-f80f4f79.json', 'project-2-at-2026-03-21-10-28-5a7e97fe.json']


In [ ]:
def flatten_export(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        tasks = json.load(f)

    rows = []
    for task in tasks:
        ann = task["annotations"][0] if task.get("annotations") else None
        label = None
        if ann:
            for r in ann.get("result", []):
                if r["from_name"] == "nli_label":
                    label = r["value"]["choices"][0]
        rows.append({
            "sample_id": task["data"]["sample_id"],
            "label": label
        })
    return pd.DataFrame(rows)

df_a = flatten_export(json_files[0]).rename(columns={"label": "label_a"})
df_b = flatten_export(json_files[1]).rename(columns={"label": "label_b"})

merged = df_a.merge(df_b, on="sample_id", how="inner")
merged

,sample_id,label_a,label_b
0,VNLI_DEMO_0003,ENT,ENT
1,VNLI_DEMO_0004,ENT,ENT
2,VNLI_DEMO_0007,ENT,ENT
3,VNLI_DEMO_0008,ENT,ENT
4,VNLI_DEMO_0010,ENT,ENT
5,VNLI_DEMO_0011,CON,CON
6,VNLI_DEMO_0013,CON,CON
7,VNLI_DEMO_0017,CON,CON
8,VNLI_DEMO_0019,CON,CON
9,VNLI_DEMO_0020,CON,CON


In [ ]:
kappa = cohen_kappa_score(merged["label_a"], merged["label_b"])
acc = (merged["label_a"] == merged["label_b"]).mean()

print("Observed agreement:", acc)
print("Cohen's kappa:", kappa)

Observed agreement: 1.0
Cohen's kappa: 1.0


In [ ]:
disagree = merged[merged["label_a"] != merged["label_b"]]
disagree

,sample_id,label_a,label_b
